# IMDb 50K Movie Reviews – NLP Project

## Step 1: Dataset Download and Loading

### 1.1 Install required libraries
Run this once. `kagglehub` downloads the dataset, `pandas` loads it as a table, `numpy` handles numbers.

In [ ]:
%pip install kagglehub pandas numpy

### 1.2 Import libraries

In [ ]:
import os
import kagglehub
import numpy as np
import pandas as pd

### 1.3 Download the dataset with KaggleHub
KaggleHub downloads the files to a cache folder on your computer and gives us the folder path.

In [ ]:
path = kagglehub.dataset_download(
    "lakshmi25npathi/imdb-dataset-of-50k-movie-reviews"
)

print("Path to dataset files:", path)

### 1.4 Find the CSV file inside the downloaded folder
We list every file in the folder and pick the one ending in `.csv`.

In [ ]:
print("Files in dataset folder:", os.listdir(path))

csv_files = [f for f in os.listdir(path) if f.endswith(".csv")]
csv_path = os.path.join(path, csv_files[0])

print("CSV file found:", csv_path)

### 1.5 Load the CSV into a Pandas DataFrame
`df` is our original dataset. We will **never modify it**; later steps will work on a copy.

In [ ]:
df = pd.read_csv(csv_path)

### 1.6 First five rows

In [ ]:
df.head()

### 1.7 Shape, column names, data types and missing values

In [ ]:
print("Shape (rows, columns):", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nMissing values in each column:")
print(df.isnull().sum())

### 1.8 Understand the `review` and `sentiment` columns
- **review**: the full text of a movie review written by a user. This is the text we will process.
- **sentiment**: the label for that review, either `positive` or `negative`.

In [ ]:
print("Unique sentiment labels:", df["sentiment"].unique())
print("\nNumber of reviews per sentiment:")
print(df["sentiment"].value_counts())

In [ ]:
print("Example review:\n")
print(df["review"].iloc[0])
print("\nSentiment:", df["sentiment"].iloc[0])

In [ ]:
# Length of each review in characters (kept in a separate variable, not added to df)
review_lengths = df["review"].str.len()

print("Review length statistics (characters):")
print(review_lengths.describe())

## Step 2: Unicode Normalization

### What is Unicode?
A computer stores every character (letter, digit, symbol, emoji) as a number. **Unicode** is the worldwide standard that decides which number belongs to which character.

### What is the problem?
The **same-looking text can be stored in different ways**. For example:
- `é` can be **one** character (`é`), or **two** characters (`e` + an accent mark `´` placed on top).
- `ﬁ` can be a single special "ligature" character instead of the two letters `f` and `i`.
- `ＩＭＤｂ` (wide "full-width" letters) looks like `IMDb`, but the computer sees different characters.
- A **non-breaking space** looks exactly like a normal space, but it is a different character.

To us they look the same, but to the computer they are **different**. So when we later count words or build Bag of Words / TF-IDF, the same word could be counted as two different words.

### What is Unicode normalization?
**Unicode normalization** converts text into **one standard form**, so that text which looks the same is also stored the same way.

There are 4 normalization forms:

| Form | What it does |
|------|--------------|
| NFC  | Combines characters (`e` + `´` → `é`) |
| NFD  | Splits characters (`é` → `e` + `´`) |
| **NFKC** | Like NFC, **and** also converts look-alike "compatibility" characters to normal ones (`ﬁ` → `fi`, `ＩＭＤｂ` → `IMDb`, non-breaking space → normal space) |
| NFKD | Like NFD, plus the compatibility conversions |

### Why NFKC?
NFKC is the most useful form for NLP on English text: it gives one standard form **and** replaces special look-alike characters with their normal versions.

NFKC does **not** lowercase the text, remove accents, remove HTML tags or remove punctuation. So the meaning of each review stays the same.

### 2.1 Import the `unicodedata` library
`unicodedata` is **built into Python**, so nothing needs to be installed. It contains the `normalize()` function.

In [ ]:
import unicodedata

### 2.2 Small examples: what does NFKC do?
Before using our dataset, we test NFKC on a few small examples.
`repr()` shows the text with invisible characters made visible (for example a non-breaking space appears as `\xa0`), and `len()` shows how many characters the text has.

In [ ]:
examples = [
    "cafe\u0301",          # 'é' written as 'e' + a separate accent mark
    "\ufb01lm",             # 'fi' written as one special ligature character
    "\uff29\uff2d\uff24\uff42",  # 'IMDb' written in full-width letters
    "great\u00a0movie",    # non-breaking space between the words
    "Wait\u2026",           # '...' written as one single character
]

for text in examples:
    normalized_text = unicodedata.normalize("NFKC", text)
    print("Original  :", repr(text), "| length:", len(text))
    print("Normalized:", repr(normalized_text), "| length:", len(normalized_text))
    print()

Now we check why this matters. Both words below look exactly like `café`, but Python says they are **not equal**. After NFKC, they become equal.

In [ ]:
word_1 = "caf\u00e9"    # 'é' as one character
word_2 = "cafe\u0301"   # 'e' + accent mark

print("word_1:", word_1, "| word_2:", word_2)
print("Equal before normalization?", word_1 == word_2)
print("Equal after normalization? ",
      unicodedata.normalize("NFKC", word_1) == unicodedata.normalize("NFKC", word_2))

### 2.3 Make a working copy of the dataset
We must **not modify the original `df`**. So we make a copy called `df_processed`. All new columns (from this step and the next steps) will be added to `df_processed`.

In [ ]:
df_processed = df.copy()

### 2.4 Create the normalization function
This function does three things:
1. Some IMDb reviewers typed the accent sign `´` instead of an apostrophe, for example `don´t`. NFKC would turn `´` into **a space + an accent mark**, which would break the word into `don ´t`. To keep the meaning, we first replace `´` with a normal apostrophe `'`.
2. In the same way, some reviewers typed the sign `¨` instead of a double quote, for example `¨director¨`. NFKC would also turn it into a space + an accent mark, so we replace `¨` with a normal double quote `"`.
3. Then it applies **NFKC** normalization.

In [ ]:
def normalize_unicode(text):
    # Replace the accent sign used as an apostrophe (don´t -> don't)
    text = text.replace("\u00b4", "'")
    # Replace the sign used as a double quote (¨director¨ -> "director")
    text = text.replace("\u00a8", '"')
    # Apply NFKC Unicode normalization
    return unicodedata.normalize("NFKC", text)

### 2.5 Apply normalization to the reviews
`apply()` runs our function on **every review** and stores the result in a **new column** called `normalized_review`.
The original `review` column stays unchanged. This is a fast operation, so it takes only a few seconds even for all 50,000 reviews.

In [ ]:
df_processed["normalized_review"] = df_processed["review"].apply(normalize_unicode)

df_processed.head()

### 2.6 How many reviews were changed?
We compare the original and normalized text of each review. `True` means the review changed.

In [ ]:
changed = df_processed["review"] != df_processed["normalized_review"]

print("Reviews changed by normalization:", changed.sum(), "out of", len(df_processed))

### 2.7 Which characters were changed?
We look inside the changed reviews and count every special character that normalization replaced.
`Counter` (from Python's built-in `collections` module) is used to count how many times each character appears.
`ord(ch) > 127` means "not a normal English keyboard character", so we only check special characters.

In [ ]:
from collections import Counter

changed_chars = Counter()

for review in df_processed.loc[changed, "review"]:
    for ch in review:
        if ord(ch) > 127 and normalize_unicode(ch) != ch:
            changed_chars[ch] += 1

print("Most common characters changed by normalization:\n")
for ch, count in changed_chars.most_common(15):
    print(repr(ch), "->", repr(normalize_unicode(ch)),
          "|", unicodedata.name(ch, "UNKNOWN"), "|", count, "times")

### 2.8 Before-and-after examples from IMDb reviews
Reviews are long, so for each example we only show a small part of the text around the first changed character.

In [ ]:
def show_change(review, window=40):
    for i, ch in enumerate(review):
        if ord(ch) > 127 and normalize_unicode(ch) != ch:
            start = max(0, i - window)
            end = i + window
            part = review[start:end]
            print("Original  :", repr(part))
            print("Normalized:", repr(normalize_unicode(part)))
            return
    print("(The change is in a combined character, for example 'e' + accent mark.)")


example_reviews = df_processed.loc[changed, "review"].head(3)

for number, review in enumerate(example_reviews, start=1):
    print("Example", number)
    show_change(review)
    print()

### 2.9 Confirm that the original dataset is unchanged
The original `df` should still have only the `review` and `sentiment` columns, and its `review` column should be exactly the same as before.

In [ ]:
print("Original df columns:   ", df.columns.tolist())
print("df_processed columns:  ", df_processed.columns.tolist())
print("Original review column unchanged:", df["review"].equals(df_processed["review"]))

## Step 3: Text Cleaning

### What is text cleaning?
Raw reviews contain things that are **not part of the actual review text**, for example:
- **HTML tags** such as `<br />`. These came from the IMDb website, where `<br />` means "new line".
- **HTML entities** such as `&amp;` (which means `&`).
- **Hidden/unwanted characters**: invisible control characters and broken characters from old Windows text encoding (for example `\x96` was meant to be a dash `-`).
- **Extra whitespace**: double spaces, tabs or spaces at the start and end.

These things add noise. For example, `br` would be counted as a "word" thousands of times.

### What we will NOT do in this step
To keep the meaning of the reviews, we do **not** lowercase the text, remove punctuation, remove numbers or remove stopwords here.

### Input and output
- Input: the `normalized_review` column from Step 2.
- Output: a **new** column called `cleaned_review` in `df_processed`.
- The original `review` column (and the original `df`) stays unchanged.

### 3.1 Import libraries
Both libraries are **built into Python**, so nothing needs to be installed.
- `re` (regular expressions): finds and replaces text **patterns**, such as "anything that looks like an HTML tag".
- `html`: converts HTML entities like `&amp;` back to normal characters like `&`.

In [ ]:
import re
import html

### 3.2 Look at the HTML tags in the reviews
Before cleaning, we check which HTML tags exist and how many reviews contain them.

Some reviewers also used `< >` in their **own writing**, for example `<grin>`, `< cough >` or `<spoiler>`. These are part of the review, not HTML, so we must **not** delete them.

So our pattern only matches **real HTML tags** that appear in this dataset:
- `<br />`, `<br>`, `<hr>` (line breaks)
- `<i>`, `</i>`, `<b>`, `</b>`, `<u>`, `<em>`, `<strong>`, `<p>` and their closing tags
- links: `<a href=...>` and `</a>`

`(?i)` means upper and lower case are both matched (`<BR>` and `<br>`).
Because the pattern only allows these exact tag names, it can never swallow normal text like `<grin>`, `< cough >`, `5 < 10` or `<3`.

In [ ]:
html_tag_pattern = r"(?i)<(?:/?(?:p|i|b|u|em|strong)|br|hr)\s*/?\s*>|<a\s+href[^<>]*>|</a\s*>"

reviews_with_tags = df_processed["normalized_review"].str.contains(html_tag_pattern, regex=True)
print("Reviews containing HTML tags:", reviews_with_tags.sum(), "out of", len(df_processed))

tag_counts = Counter()
for review in df_processed.loc[reviews_with_tags, "normalized_review"]:
    for tag in re.findall(html_tag_pattern, review):
        tag_counts[tag.lower()] += 1

print("\nMost common HTML tags:")
for tag, count in tag_counts.most_common(10):
    print(repr(tag), "|", count, "times")

### 3.3 Look at the unwanted characters in the reviews
- **Control characters** (`\x00`–`\x1f` and `\x7f`–`\x9f`) are invisible characters that are not real text. In this dataset, some of them are broken characters from old Windows encoding. For example, `\x96` was originally a dash and `\x85` was originally `...`.
- **Invisible format characters**, such as the zero-width space `\u200b`, take no space on screen but can join or split words.

In [ ]:
control_char_pattern = r"[\x00-\x1f\x7f-\x9f]"
invisible_char_pattern = r"[\u00ad\u200b-\u200f\u202a-\u202e\u2060-\u2064\ufeff]"

unwanted_counts = Counter()
for review in df_processed["normalized_review"]:
    for ch in re.findall(control_char_pattern + "|" + invisible_char_pattern, review):
        unwanted_counts[ch] += 1

print("Unwanted characters found:\n")
for ch, count in unwanted_counts.most_common(15):
    print(repr(ch), "|", unicodedata.name(ch, "CONTROL CHARACTER"), "|", count, "times")

### 3.4 Create the cleaning functions
We write one small function for each cleaning task, so each part is easy to understand.

**1. `remove_html_tags()`** replaces every HTML tag with a **space**. We use a space (not nothing) so that words on both sides of `<br />` do not get stuck together, for example `end.<br /><br />The` → `end. The`.

**2. `fix_html_entities()`** converts HTML entities like `&amp;` → `&` and `&quot;` → `"`.

**3. `fix_unwanted_characters()`**
- First, it changes the broken old-Windows characters back to what the writer meant (`\x96` → `-`, `\x85` → `...`, `\x92` → `'`, and letters in names such as `\x8e` → `Ž`), so the meaning is kept.
- Then, it replaces any remaining control characters with a space, and removes invisible format characters.

**4. `remove_extra_whitespace()`** changes many spaces, tabs or new lines into **one** space, and removes spaces at the start and end.

**5. `clean_text()`** runs all four functions in the correct order.

In [ ]:
# Broken characters from old Windows encoding -> what the writer meant
broken_character_fixes = {
    "\x80": "€",     # euro sign
    "\x84": '"',     # low double quote
    "\x85": "...",   # ellipsis
    "\x8a": "Š",     # letter S with caron (in names)
    "\x8e": "Ž",     # letter Z with caron (in names)
    "\x91": "'",     # left single quote
    "\x92": "'",     # right single quote / apostrophe
    "\x93": '"',     # left double quote
    "\x94": '"',     # right double quote
    "\x96": "-",     # en dash
    "\x97": "-",     # em dash
    "\x9a": "š",     # letter s with caron (in names)
    "\x9e": "ž",     # letter z with caron (in names)
}


def remove_html_tags(text):
    return re.sub(html_tag_pattern, " ", text)


def fix_html_entities(text):
    return html.unescape(text)


def fix_unwanted_characters(text):
    for broken, fixed in broken_character_fixes.items():
        text = text.replace(broken, fixed)
    text = re.sub(control_char_pattern, " ", text)
    text = re.sub(invisible_char_pattern, "", text)
    return text


def remove_extra_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()


def clean_text(text):
    text = remove_html_tags(text)
    text = fix_html_entities(text)
    text = fix_unwanted_characters(text)
    text = remove_extra_whitespace(text)
    return text

### 3.5 Test the cleaning function on a small example
Before using it on the dataset, we test it on one made-up sentence that has all the problems.

In [ ]:
test_text = "Great movie!<br /><br />The   acting was good \x96 not perfect.\x85 Tom &amp; Jerry   "

print("Before:", repr(test_text))
print("After: ", repr(clean_text(test_text)))

### 3.6 Apply cleaning to the reviews
We apply `clean_text()` to the `normalized_review` column and save the result in a **new column** called `cleaned_review`.

In [ ]:
df_processed["cleaned_review"] = df_processed["normalized_review"].apply(clean_text)

df_processed.head()

### 3.7 Before-and-after examples
Example 1 shows the first 300 characters of a review with `<br />` tags.
For the other examples, the change can be anywhere in a long review, so `show_around()` shows only the small part of the text **around** the change.

In [ ]:
def show_before_after(index, n_chars=300):
    print("Review index:", index)
    print("BEFORE:", df_processed.loc[index, "normalized_review"][:n_chars])
    print()
    print("AFTER: ", df_processed.loc[index, "cleaned_review"][:n_chars])
    print("-" * 80)


def show_around(pattern, title, window=60):
    # Find the first review that matches the pattern, and show the text around the match
    for index, review in df_processed["normalized_review"].items():
        match = re.search(pattern, review)
        if match:
            start = max(0, match.start() - window)
            end = match.end() + window
            part = review[start:end]
            print(title)
            print("Review index:", index)
            print("BEFORE:", repr(part))
            print("AFTER: ", repr(clean_text(part)))
            print("-" * 80)
            return
    print(title, "-> no review found")


# Example 1: a review with <br /> tags
show_before_after(1)

# Example 2: a broken old-Windows character (dash)
show_around("\x96", "Example 2: broken dash character")

# Example 3: a broken old-Windows character (ellipsis)
show_around("\x85", "Example 3: broken ellipsis character")

# Example 4: an HTML entity such as &amp;
show_around(r"&(?:amp|quot|lt|gt|#\d+);", "Example 4: HTML entity")

# Example 5: the reviewer's own < > text is kept, not deleted
show_around(r"(?i)< cough >", "Example 5: reviewer's own < > text")

### 3.8 Check the cleaning results
After cleaning, there should be **0** reviews with HTML tags, **0** with control characters and **0** with extra spaces.
We also compare the average review length before and after cleaning.

In [ ]:
cleaned = df_processed["cleaned_review"]

print("Reviews still containing HTML tags:      ", cleaned.str.contains(html_tag_pattern, regex=True).sum())
print("Reviews still containing control chars:  ", cleaned.str.contains(control_char_pattern, regex=True).sum())
print("Reviews still containing double spaces:  ", cleaned.str.contains("  ", regex=False).sum())
print("Empty reviews after cleaning:            ", (cleaned.str.len() == 0).sum())

print("\nAverage length before cleaning:", round(df_processed["normalized_review"].str.len().mean(), 1), "characters")
print("Average length after cleaning: ", round(cleaned.str.len().mean(), 1), "characters")

### 3.9 Confirm that the original dataset is unchanged

In [ ]:
print("Original df columns:   ", df.columns.tolist())
print("df_processed columns:  ", df_processed.columns.tolist())
print("Original review column unchanged:", df["review"].equals(df_processed["review"]))

## Step 4: Regular Expressions (Regex)

### What are Regular Expressions?
A **regular expression (regex)** is a short "search pattern" that describes **what kind of text** we are looking for.
Instead of searching for one exact word, we can search for a **type** of text, for example:
- "any number" → `\d+`
- "any web link" → `https?://\S+`
- "the same `!` repeated many times" → `!{2,}`

### Why is regex important in NLP?
Real text is messy. Regex lets us **find, count, replace or remove** patterns in thousands of reviews with one line of code:
- remove things that are not useful words (URLs, email addresses, strange symbols),
- make similar things look the same (`!!!!!` → `!`, `` it`s `` → `it's`),
- find useful information (for example ratings like `7/10`).

### Regex symbols used in this step

| Symbol | Meaning | Example |
|--------|---------|---------|
| `\d` | any digit 0–9 | `\d+` matches `2006` |
| `\w` | any letter, digit or `_` (also accented letters like `é`) | `\w+` matches `Prévert` |
| `\s` | any whitespace (space, tab, new line) | |
| `\S` | anything that is **not** whitespace | `\S+` matches `www.imdb.com` |
| `+` | one or more times | `\d+` |
| `{2,}` | 2 or more times | `!{2,}` matches `!!!` |
| `[...]` | any **one** character from the list | `[!?]` matches `!` or `?` |
| `[^...]` | any **one** character **not** in the list | `[^a-z]` |
| `\|` (written `|`) | OR | `http|www` |
| `( )` and `\1` | remember a character, then "the same character again" | `([!?])\1+` |
| `\b` | word boundary (start or end of a word) | `\b\d+/10\b` |
| `\.` | a real dot (a plain `.` means "any character") | `www\.` |

### Input and output
- Input: the `cleaned_review` column from Step 3.
- Output: a **new** column called `regex_review`.
- We do **not** lowercase the text here (that is Step 5) and we do not split it into words (that is Step 6).

### 4.1 A small regex demonstration
The `re` library was already imported in Step 3, so we can use it directly.
- `re.findall(pattern, text)` returns a **list of everything** that matches the pattern.
- `re.sub(pattern, replacement, text)` **replaces** everything that matches the pattern.

In [ ]:
demo_text = "I gave it 9/10!!! Visit www.example.com or http://imdb.com for more... It`s great *****"

print("Text:", demo_text)
print()
print("Digits          \\d+           :", re.findall(r"\d+", demo_text))
print("Words           [A-Za-z]+     :", re.findall(r"[A-Za-z]+", demo_text))
print("Punctuation     [^\\w\\s]        :", re.findall(r"[^\w\s]", demo_text))
print("URLs            https?://\\S+|www\\.\\S+ :", re.findall(r"https?://\S+|www\.\S+", demo_text))
print("Ratings         \\b\\d{1,2}/10\\b  :", re.findall(r"\b\d{1,2}/10\b", demo_text))
print("Repeated ! or ? [!?]{2,}       :", re.findall(r"[!?]{2,}", demo_text))

### 4.2 Find these patterns in the IMDb reviews
Before changing anything, we **count** how many cleaned reviews contain each pattern.
`count_reviews()` uses `re.search()` on every review. It gives `True` if the pattern is found in that review, and `.sum()` counts the `True` values.

In [ ]:
url_pattern = r"https?://\S+|www\.\S+"
email_pattern = r"\S+@\S+\.\S+"
odd_apostrophe_pattern = r"[`‘’]"
odd_double_quote_pattern = r"[“”]"
repeated_punctuation_pattern = r"([!?,;:\-])\1+"
long_dots_pattern = r"\.{4,}"
heart_pattern = r"<3+"
special_char_pattern = r"[^\w\s.,!?'\"():;\-/&%$]|_"
digit_pattern = r"\d+"
rating_pattern = r"\b\d{1,2}/10\b"

patterns_to_check = {
    "URLs": url_pattern,
    "Email addresses": email_pattern,
    "Odd apostrophes ` ‘ ’": odd_apostrophe_pattern,
    "Curly double quotes “ ”": odd_double_quote_pattern,
    "Repeated punctuation (!!, ??, --)": repeated_punctuation_pattern,
    "Long dots (....)": long_dots_pattern,
    "Heart symbol <3": heart_pattern,
    "Special characters (* # ~ ^ | _ ...)": special_char_pattern,
    "Digits": digit_pattern,
    "Ratings like 7/10": rating_pattern,
}

def count_reviews(column, pattern):
    # re.search() returns a match if the pattern is found anywhere in the review
    return df_processed[column].apply(lambda text: re.search(pattern, text) is not None).sum()


for name, pattern in patterns_to_check.items():
    count = count_reviews("cleaned_review", pattern)
    print(f"{name:40} -> found in {count} reviews")

Now we look at a few **real matches** for each pattern, so we can see what they look like in the dataset.

In [ ]:
def show_matches(pattern, name, n_examples=8):
    matches = []
    for review in df_processed["cleaned_review"]:
        for match in re.finditer(pattern, review):
            matches.append(match.group())
            if len(matches) == n_examples:
                print(f"{name}: {matches}")
                return
    print(f"{name}: {matches}")


show_matches(url_pattern, "URLs")
show_matches(email_pattern, "Email addresses")
show_matches(repeated_punctuation_pattern, "Repeated punctuation")
show_matches(special_char_pattern, "Special characters")
show_matches(rating_pattern, "Ratings")

### 4.3 Decide what to do with each pattern
To **preserve meaningful English words and sentence content**, we handle each pattern as follows:

| Pattern | Regex | What we do | Why |
|---------|-------|------------|-----|
| URLs | `https?://\S+\|www\.\S+` | **remove** (replace with a space) | a web link is not a word and says nothing about the movie's sentiment |
| Email addresses | `\S+@\S+\.\S+` | **remove** | same reason as URLs |
| Odd apostrophes | `` [`‘’] `` | **replace with** `'` | `` it`s ``, `it’s` and `it's` are the same word; keep it one form |
| Curly double quotes | `[“”]` | **replace with** `"` | same idea |
| Repeated punctuation | `([!?,;:\-])\1+` | **reduce to one** (`!!!` → `!`) | `(...)` remembers the character and `\1+` means "the same character again, one or more times" |
| Long dots | `\.{4,}` | **replace with** `...` | 4 or more dots become a normal ellipsis |
| Heart symbol | `<3+` | **remove** | `<3` is a heart drawn with symbols. If we only removed `<`, the heart would wrongly become the number `3` |
| Special characters | `[^\w\s.,!?'"():;\-/&%$]\|_` | **remove** (replace with a space) | symbols like `*`, `#`, `~`, `^`, `\|`, `_`, `<`, `>`, `=` carry no word meaning. `[^...]` means "every character **except** the ones in the list", so letters (also `é`, `ï`), digits, spaces and normal sentence punctuation are **kept** |
| Extra spaces | `\s+` | **reduce to one space** | removing things above can leave double spaces |

**What we keep, and why:**
- **Normal punctuation** (`. , ! ? ' " ( ) : ; -`) is kept, because it shows where sentences end and keeps words like `don't` and `old-time` correct.
- **Digits** are kept, because numbers like `7/10`, `1/10` or `1950s` carry real meaning in movie reviews (a `1/10` rating is very negative).
- `/`, `&`, `%`, `$` are kept for `7/10`, `A&E`, `100%` and `$5`.

### 4.4 Create the regex processing function
`regex_process()` applies every rule from the table above, **in order**.
URLs and emails are removed **first**, before the special-character rule removes `@`, `:` or `/` from inside them.

In [ ]:
def remove_urls(text):
    return re.sub(url_pattern, " ", text)


def remove_emails(text):
    return re.sub(email_pattern, " ", text)


def standardize_quotes(text):
    text = re.sub(odd_apostrophe_pattern, "'", text)
    text = re.sub(odd_double_quote_pattern, '"', text)
    return text


def reduce_repeated_punctuation(text):
    text = re.sub(long_dots_pattern, "...", text)
    text = re.sub(repeated_punctuation_pattern, r"\1", text)
    return text


def remove_special_characters(text):
    text = re.sub(heart_pattern, " ", text)
    text = re.sub(special_char_pattern, " ", text)
    return text


def regex_process(text):
    text = remove_urls(text)
    text = remove_emails(text)
    text = standardize_quotes(text)
    text = reduce_repeated_punctuation(text)
    text = remove_special_characters(text)
    text = remove_extra_whitespace(text)   # function from Step 3
    return text

### 4.5 Test the function on a small example

In [ ]:
test_text = "WOW!!!! It`s a 10/10 movie..... Visit www.example.com or mail me at fan@mail.com ***** #best ~~great~~"

print("Before:", repr(test_text))
print("After: ", repr(regex_process(test_text)))

### 4.6 Apply regex processing to the reviews
We apply `regex_process()` to the `cleaned_review` column and save the result in a **new column** called `regex_review`.

In [ ]:
df_processed["regex_review"] = df_processed["cleaned_review"].apply(regex_process)

df_processed[["cleaned_review", "regex_review"]].head()

### 4.7 Before-and-after examples from IMDb reviews
For each pattern, we find the first review that contains it and show the text **around** the match, before and after regex processing.

In [ ]:
def show_regex_change(pattern, title, window=60):
    for index, review in df_processed["cleaned_review"].items():
        match = re.search(pattern, review)
        if match:
            start = max(0, match.start() - window)
            end = match.end() + window
            part = review[start:end]
            print(title)
            print("Review index:", index)
            print("BEFORE:", repr(part))
            print("AFTER: ", repr(regex_process(part)))
            print("-" * 80)
            return
    print(title, "-> no review found")


show_regex_change(url_pattern, "Example 1: URL removed")
show_regex_change(r"!{2,}", "Example 2: repeated '!' reduced")
show_regex_change(long_dots_pattern, "Example 3: long dots reduced to '...'")
show_regex_change(odd_apostrophe_pattern, "Example 4: odd apostrophe standardized")
show_regex_change(r"\*", "Example 5: special characters (*) removed")
show_regex_change(rating_pattern, "Example 6: rating kept")

### 4.8 Why we did NOT remove digits and punctuation
Regex **can** remove all digits (`\d+`) or all punctuation (`[^\w\s]`) in one line. Below we only **demonstrate** it on one review, without saving it, so you can see why we kept them.

In [ ]:
index = df_processed["regex_review"].apply(lambda text: re.search(rating_pattern, text) is not None).idxmax()
match = re.search(rating_pattern, df_processed.loc[index, "regex_review"])
part = df_processed.loc[index, "regex_review"][max(0, match.start() - 80): match.end() + 40]

print("Regex output (kept):      ", repr(part))
print("If digits were removed:   ", repr(re.sub(r"\d+", "", part)))
print("If punctuation was removed:", repr(re.sub(r"[^\w\s]", "", part)))

Removing digits loses the rating, and removing punctuation breaks words like `don't` → `dont`. So for this step, keeping them is the better choice.

### 4.9 Check the regex results

In [ ]:
print("Reviews still containing URLs:                ", count_reviews("regex_review", url_pattern))
print("Reviews still containing emails:              ", count_reviews("regex_review", email_pattern))
print("Reviews still containing special characters:  ", count_reviews("regex_review", special_char_pattern))
print("Reviews still containing repeated punctuation:", count_reviews("regex_review", repeated_punctuation_pattern))
print("Reviews still containing double spaces:       ", count_reviews("regex_review", r"  "))
print("Empty reviews after regex processing:         ", (df_processed["regex_review"].str.len() == 0).sum())
print("Reviews changed by regex processing:          ", (df_processed["cleaned_review"] != df_processed["regex_review"]).sum())

print("\nOriginal review column unchanged:", df["review"].equals(df_processed["review"]))
print("df_processed columns:", df_processed.columns.tolist())

## Step 5: Lowercase Normalization

### What is lowercase normalization?
It means converting **every capital letter into a small letter**, for example `GREAT Movie` → `great movie`.

### Why is it important in NLP?
For a computer, `Great`, `GREAT` and `great` are **three different words**, even though they mean the same thing.
- A word at the start of a sentence is capitalized (`Great acting.`), and the same word in the middle is not (`It was great.`).
- Reviewers often use CAPITALS for emphasis (`This movie was AWFUL`).

If we do not lowercase, later steps such as word frequency, Bag of Words and TF-IDF would count these as separate words. This makes the vocabulary bigger and the counts less accurate. After lowercasing, all forms become **one word**.

### A small downside
Some information is lost, for example `US` (the country) and `us` (the word) become the same. For sentiment analysis of movie reviews this is a small loss, and the benefit is much bigger.

### Input and output
- Input: the `regex_review` column from Step 4.
- Output: a **new** column called `lowercase_review`.

### 5.1 A small example
Python's built-in string method `.lower()` converts text to lowercase. No library is needed.

In [ ]:
words = ["Great", "GREAT", "great", "GrEaT"]

print("Before lowercasing:", words, "-> unique words:", len(set(words)))

lower_words = [word.lower() for word in words]
print("After lowercasing: ", lower_words, "-> unique words:", len(set(lower_words)))

`set()` keeps only the **unique** values. Before lowercasing there are 4 different words; after lowercasing there is only 1.

### 5.2 Apply lowercasing to the reviews
`str.lower()` lowercases **every review** in the column at once. We save the result in a **new column** called `lowercase_review`.

In [ ]:
df_processed["lowercase_review"] = df_processed["regex_review"].str.lower()

df_processed[["regex_review", "lowercase_review"]].head()

### 5.3 Before-and-after examples
Example 1 is the first review. Example 2 is the first review that contains a word written fully in CAPITALS (4 or more capital letters).
The regex `\b[A-Z]{4,}\b` means: a whole word (`\b` ... `\b`) made of 4 or more capital letters (`[A-Z]{4,}`).

In [ ]:
def show_lowercase_change(index, n_chars=250):
    print("Review index:", index)
    print("BEFORE:", df_processed.loc[index, "regex_review"][:n_chars])
    print("AFTER: ", df_processed.loc[index, "lowercase_review"][:n_chars])
    print("-" * 80)


print("Example 1: first review")
show_lowercase_change(0)

print("Example 2: a review with a word in CAPITALS")
capital_pattern = r"\b[A-Z]{4,}\b"
capital_index = df_processed["regex_review"].apply(lambda text: re.search(capital_pattern, text) is not None).idxmax()
match = re.search(capital_pattern, df_processed.loc[capital_index, "regex_review"])
start = max(0, match.start() - 80)
end = match.end() + 80
print("Review index:", capital_index)
print("CAPITAL word found:", match.group())
print("BEFORE:", df_processed.loc[capital_index, "regex_review"][start:end])
print("AFTER: ", df_processed.loc[capital_index, "lowercase_review"][start:end])
print("-" * 80)

### 5.4 The difference in the whole dataset
We count how many times different spellings of the word **great** appear, before and after lowercasing.
`\bgreat\b` matches the whole word `great` only (not `greatest`). `re.findall()` returns all matches and `len()` counts them.

In [ ]:
def count_word(column, word):
    pattern = r"\b" + word + r"\b"
    return df_processed[column].apply(lambda text: len(re.findall(pattern, text))).sum()


print("BEFORE lowercasing (regex_review):")
for word in ["great", "Great", "GREAT"]:
    print(f"  '{word}': {count_word('regex_review', word)} times")

print("\nAFTER lowercasing (lowercase_review):")
for word in ["great", "Great", "GREAT"]:
    print(f"  '{word}': {count_word('lowercase_review', word)} times")

After lowercasing, `Great` and `GREAT` appear **0** times, and all of them are now counted as `great`.

### 5.5 Check the lowercase results
- No review should still contain capital letters. We check this by comparing each review with its own `.lower()` version: if they are the same, there are no capital letters left (this also covers capital letters with accents, like `É`).
- The text should be the same length as before, because lowercasing only changes letters and does not add or remove any.

In [ ]:
lowercase_text = df_processed["lowercase_review"]

print("Reviews still containing capital letters:", (lowercase_text != lowercase_text.str.lower()).sum())
print("Reviews whose length changed:            ", (lowercase_text.str.len() != df_processed["regex_review"].str.len()).sum())
print("Empty reviews:                           ", (lowercase_text.str.len() == 0).sum())

print("\nOriginal review column unchanged:", df["review"].equals(df_processed["review"]))
print("df_processed columns:", df_processed.columns.tolist())

## Step 6: Tokenization

### What is tokenization?
**Tokenization** means breaking text into small pieces called **tokens**. In **word tokenization**, each token is a word or a punctuation mark.

Example: `"i loved it!"` → `['i', 'loved', 'it', '!']`

### Why is it required in NLP?
A computer cannot "see" words inside a long piece of text. Almost every next NLP step works on **individual words**, not on one long string:
- removing stopwords (remove the word `the`),
- stemming and lemmatization (change `movies` → `movie`),
- word frequency, N-grams, Bag of Words and TF-IDF (count words).

So tokenization is the step that turns each review into a **list of words** that later steps can use.

### Why use NLTK instead of `split()`?
Python's `text.split()` only cuts the text at spaces, so punctuation stays stuck to words (`it!`, `movie.`). NLTK's `word_tokenize()` understands English: it separates punctuation and also splits contractions (`don't` → `do` + `n't`).

### Input and output
- Input: the `lowercase_review` column from Step 5.
- Output: a **new** column called `tokens`, where each value is a **list of tokens**.

### 6.1 Install and import NLTK
**NLTK (Natural Language Toolkit)** is the most common Python library for learning NLP. We need it here for `word_tokenize()`.
- `%pip install nltk` installs it (run once; Anaconda usually already has it).
- `word_tokenize()` needs a small English data package called **punkt**, so we download it with `nltk.download()`. Newer NLTK versions use `punkt_tab`, so we download both. `quiet=True` hides the long download messages.

In [ ]:
%pip install nltk

In [ ]:
import nltk
from nltk.tokenize import word_tokenize

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

print("NLTK version:", nltk.__version__)

### 6.2 A small example: `split()` vs `word_tokenize()`

In [ ]:
example_text = "i don't think it's a great movie... i'd give it 3/10! the old-time feel was nice, though."

print("Text:", example_text)
print()
print("split()        :", example_text.split())
print()
print("word_tokenize():", word_tokenize(example_text))

**How to read the output:**
- With `split()`, punctuation stays attached: `movie...`, `3/10!`, `nice,`, `though.`. The computer would treat `movie...` and `movie` as different words.
- With `word_tokenize()`:
  - punctuation becomes its own token: `...`, `!`, `,`, `.`
  - contractions are split into two meaningful parts: `don't` → `do` + `n't`, `it's` → `it` + `'s`, `i'd` → `i` + `'d`. The token `n't` keeps the meaning **not**, which is important for sentiment.
  - `3/10` and `old-time` stay as one token, so the rating and the hyphenated word are kept.

### 6.3 Apply tokenization to the reviews
Some reviewers typed a dash directly after a word, for example `fan- always`. NLTK then gives the token `fan-`, which the computer treats as a different word from `fan`.
So `tokenize_review()` first runs NLTK's `word_tokenize()`, and then removes a hyphen stuck at the **start or end** of a token with `strip("-")`. Hyphens **inside** words stay, so `old-time` is kept. A token that is only a dash (`-`) is also kept, because stripping it would leave nothing.

`apply(tokenize_review)` runs this on **every** lowercase review and saves the list of tokens in a **new column** called `tokens`.

**Note:** this is the slowest step so far. For all 50,000 reviews it can take about **1–2 minutes**. Wait until `[*]` changes to a number.

In [ ]:
def tokenize_review(text):
    tokens = word_tokenize(text)
    # "fan- always" gives the token "fan-", so remove a hyphen stuck at the start or end of a token
    return [token.strip("-") if token.strip("-") else token for token in tokens]


df_processed["tokens"] = df_processed["lowercase_review"].apply(tokenize_review)

df_processed[["lowercase_review", "tokens"]].head()

### 6.4 Examples: original review and its tokens
For each example we show:
1. the **original review** (from the untouched `review` column),
2. the **lowercase review** (the input of tokenization),
3. the **tokens** (the output of tokenization).

Reviews are long, so we show only the first 300 characters and the first 40 tokens.

In [ ]:
def show_tokens(index, n_chars=300, n_tokens=40):
    tokens = df_processed.loc[index, "tokens"]
    print("Review index:", index)
    print("ORIGINAL REVIEW: ", df_processed.loc[index, "review"][:n_chars])
    print()
    print("LOWERCASE REVIEW:", df_processed.loc[index, "lowercase_review"][:n_chars])
    print()
    print("TOKENS (first", n_tokens, "):", tokens[:n_tokens])
    print()
    print("Total tokens in this review:", len(tokens))
    print("-" * 80)


# Example 1 and 2: the first two reviews
show_tokens(0)
show_tokens(1)

# Example 3: the first review that contains a rating like 7/10
rating_index = df_processed["lowercase_review"].apply(lambda text: re.search(rating_pattern, text) is not None).idxmax()
show_tokens(rating_index)

# The rating is usually later in the review, so also show the part of the text around it
text = df_processed.loc[rating_index, "lowercase_review"]
match = re.search(rating_pattern, text)
part = text[max(0, match.start() - 60): match.end() + 60]
print("Part of review", rating_index, "around the rating", match.group())
print("TEXT:  ", part)
print("TOKENS:", tokenize_review(part))

### 6.5 How contractions and ratings are tokenized in the dataset
Here we look at a short part of a real review that contains `n't` (from words like `don't`, `didn't`), to see how it was split.

In [ ]:
contraction_index = df_processed["lowercase_review"].apply(lambda text: "n't" in text).idxmax()
text = df_processed.loc[contraction_index, "lowercase_review"]
position = text.index("n't")
part = text[max(0, position - 60): position + 60]

print("Review index:", contraction_index)
print("TEXT:  ", part)
print("TOKENS:", tokenize_review(part))

### 6.6 Token statistics
- **Tokens per review**: how long the reviews are, counted in tokens.
- **Total tokens**: all tokens in all reviews together.
- **Unique tokens**: how many **different** tokens exist (the size of the vocabulary). `set()` keeps each token only once.

In [ ]:
token_counts = df_processed["tokens"].apply(len)

print("Tokens per review:")
print(token_counts.describe().round(1))

unique_tokens = set()
for tokens in df_processed["tokens"]:
    unique_tokens.update(tokens)

print("\nTotal tokens in all reviews: ", token_counts.sum())
print("Unique tokens (vocabulary):  ", len(unique_tokens))

### 6.7 Check the tokenization results
- Every value in `tokens` should be a **list**.
- No review should have an empty token list.
- No token should contain a space (spaces are only used to separate tokens).

In [ ]:
print("All values are lists:          ", df_processed["tokens"].apply(lambda tokens: isinstance(tokens, list)).all())
print("Reviews with no tokens:         ", (token_counts == 0).sum())
print("Tokens containing a space:      ", sum(1 for token in unique_tokens if " " in token))

print("\nOriginal review column unchanged:", df["review"].equals(df_processed["review"]))
print("df_processed columns:", df_processed.columns.tolist())

## Step 7: Stop Words Removal

### What are stop words?
**Stop words** are very common words such as `the`, `is`, `a`, `and`, `of`, `to`, `it`, `this`.
They appear in almost every sentence, but on their own they carry **very little meaning** about the topic or the sentiment of a review.

### Why are they removed in NLP?
- They are the **most frequent** words, so they hide the important words (`brilliant`, `boring`, `acting`) in word counts.
- Removing them makes each review **shorter**, so later steps run faster and use less memory.
- Models such as Bag of Words and TF-IDF can then focus on the words that actually describe the movie.

### Why we must keep negation words
NLTK's standard stopword list also contains **negation words** like `not`, `no`, `nor`, `don't` and `didn't`.
In sentiment analysis these words are very important, because they **reverse the meaning**:
- `this movie is not good` → without negations becomes `movie good` → looks **positive**, but the review is **negative**.
- `i did n't like it` → without negations becomes `like` → looks **positive**, but the review is **negative**.

So we use NLTK's list, but we **remove the negation words from the stopword list** before removing stopwords from the reviews.

### Input and output
- Input: the `tokens` column from Step 6.
- Output: a **new** column called `tokens_no_stopwords`.
- Punctuation tokens (`.`, `,`, `!`) are not stop words, so they stay in this step.

### 7.1 Download and load NLTK's English stopword list
`nltk.download("stopwords")` downloads the stopword lists (only the first time). `stopwords.words("english")` gives the English list.
We store it as a **set**, because checking "is this word in the set?" is much faster than checking a normal list. This matters because we will check more than 13 million tokens.

In [ ]:
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

english_stopwords = set(stopwords.words("english"))

print("Number of English stopwords in NLTK:", len(english_stopwords))
print("\nSome stopwords:", sorted(english_stopwords)[:40])

### 7.2 Keep the negation words
`negation_words` contains every word that means "not/no".
- `not`, `no`, `nor`, `never`, `none`, `nothing`, `nobody`, `nowhere`, `neither`, `cannot`
- `n't`: in Step 6, NLTK split `don't` into `do` + `n't`, so `n't` is the token that carries the "not" meaning.
- NLTK's list also contains short forms such as `don`, `didn`, `isn`, `wasn`. These appear when a reviewer writes `dont` or `isnt` without an apostrophe, so we keep them too.

Then `stop_words_to_remove` is the NLTK list **without** the negation words. We also keep every stopword that ends with `n't`, such as `don't` and `wasn't`.

In [ ]:
negation_words = {
    "not", "no", "nor", "never", "none", "nothing", "nobody", "nowhere", "neither", "cannot", "n't",
    "ain", "aren", "couldn", "didn", "doesn", "don", "hadn", "hasn", "haven", "isn",
    "mightn", "mustn", "needn", "shan", "shouldn", "wasn", "weren", "won", "wouldn",
}

stop_words_to_remove = {
    word for word in english_stopwords
    if word not in negation_words and not word.endswith("n't")
}

kept_from_nltk_list = sorted(english_stopwords - stop_words_to_remove)

print("Stopwords in NLTK list:     ", len(english_stopwords))
print("Stopwords we will remove:   ", len(stop_words_to_remove))
print("Negation words we KEEP:     ", len(kept_from_nltk_list))
print("\nNegation words kept from the NLTK list:")
print(kept_from_nltk_list)

### 7.3 Small example: why keeping negations matters
We remove stopwords from two small token lists in two ways:
1. with the **full** NLTK list (negations are removed),
2. with **our** list (negations are kept).

In [ ]:
example_reviews = [
    ["this", "movie", "is", "not", "good", "."],
    ["i", "do", "not", "like", "the", "acting", "and", "no", "one", "should", "watch", "it", "."],
]

for tokens in example_reviews:
    with_full_list = [token for token in tokens if token not in english_stopwords]
    with_our_list = [token for token in tokens if token not in stop_words_to_remove]
    print("Tokens:                    ", tokens)
    print("Full NLTK list (WRONG):    ", with_full_list)
    print("Our list, negations kept:  ", with_our_list)
    print()

With the full list, both negative reviews end up looking **positive** (`movie good`, `like acting one watch`). With our list, `not` and `no` stay, so the negative meaning is kept.

### 7.4 Remove stopwords from all reviews
`remove_stopwords()` keeps every token that is **not** in `stop_words_to_remove`.
This is a **list comprehension**: `[token for token in tokens if ...]` builds a new list from the tokens that pass the check.
It creates a **new** list, so the `tokens` column from Step 6 is not changed.

In [ ]:
def remove_stopwords(tokens):
    return [token for token in tokens if token not in stop_words_to_remove]


df_processed["tokens_no_stopwords"] = df_processed["tokens"].apply(remove_stopwords)

df_processed[["tokens", "tokens_no_stopwords"]].head()

### 7.5 Before-and-after examples
For each example we show the first 30 tokens **before**, the stopwords that were **removed** from them, and the tokens that **remain**.

In [ ]:
def show_stopword_removal(index, n_tokens=30):
    before = df_processed.loc[index, "tokens"][:n_tokens]
    removed = [token for token in before if token in stop_words_to_remove]
    remaining = [token for token in before if token not in stop_words_to_remove]
    print("Review index:", index)
    print("BEFORE (first", n_tokens, "tokens):", before)
    print()
    print("REMOVED STOPWORDS:", removed)
    print()
    print("REMAINING TOKENS: ", remaining)
    print()
    print("Tokens in whole review:", len(df_processed.loc[index, "tokens"]),
          "->", len(df_processed.loc[index, "tokens_no_stopwords"]))
    print("-" * 80)


# Example 1 and 2: the first two reviews
show_stopword_removal(0)
show_stopword_removal(1)

Example 3 shows a short part of a real review that contains `not`, so we can see that the negation is **kept**.

In [ ]:
not_index = df_processed["tokens"].apply(lambda tokens: "not" in tokens).idxmax()
tokens = df_processed.loc[not_index, "tokens"]
position = tokens.index("not")
part = tokens[max(0, position - 10): position + 10]

print("Review index:", not_index)
print("BEFORE:", part)
print("AFTER: ", remove_stopwords(part))

### 7.6 Statistics: how many tokens were removed?
`Counter` (imported in Step 2) counts how many times each removed stopword appeared, so we can see the most common ones.

In [ ]:
tokens_before = df_processed["tokens"].apply(len).sum()
tokens_after = df_processed["tokens_no_stopwords"].apply(len).sum()

print("Total tokens before stopword removal:", tokens_before)
print("Total tokens after stopword removal: ", tokens_after)
print("Tokens removed:                      ", tokens_before - tokens_after,
      f"({(tokens_before - tokens_after) / tokens_before * 100:.1f}% of all tokens)")

removed_counts = Counter()
for tokens in df_processed["tokens"]:
    for token in tokens:
        if token in stop_words_to_remove:
            removed_counts[token] += 1

print("\nMost common removed stopwords:")
for word, count in removed_counts.most_common(15):
    print(f"  {word:8} -> {count} times")

### 7.7 Check the results
- No token in `tokens_no_stopwords` should be in `stop_words_to_remove`.
- The negation words must appear **the same number of times** before and after, which proves they were not removed.
- The `tokens` column from Step 6 must still have the same number of tokens as before (13,191,275), which proves it was not changed.

In [ ]:
stopwords_left = df_processed["tokens_no_stopwords"].apply(
    lambda tokens: sum(1 for token in tokens if token in stop_words_to_remove)
).sum()
print("Stopwords still left:", stopwords_left)

print("\nNegation words (before -> after):")
for word in ["not", "no", "never", "n't", "nothing"]:
    before = df_processed["tokens"].apply(lambda tokens: tokens.count(word)).sum()
    after = df_processed["tokens_no_stopwords"].apply(lambda tokens: tokens.count(word)).sum()
    print(f"  {word:8} {before} -> {after}")

print("\nReviews with no tokens left:", (df_processed["tokens_no_stopwords"].apply(len) == 0).sum())
print("Tokens in Step 6 column (unchanged):", df_processed["tokens"].apply(len).sum())

print("\nOriginal review column unchanged:", df["review"].equals(df_processed["review"]))
print("df_processed columns:", df_processed.columns.tolist())